In [ ]:
import sys
import pickle
import numpy as np
import pandas as pd
from itertools import combinations
from math import factorial
from collections import defaultdict
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.stattools import adfuller
from statsmodels.regression.linear_model import RegressionResultsWrapper 

sys.path.append('..')
from modules.source import Sources, parquet_monthly, PROCESSED_DATA_PATH

CHANNELS: dict[str, tuple[str]] = {
  "Domestic Baseline": (Sources.cpi_inflation, Sources.IPI),
  "Rate Differential": (Sources.rate_differential,),
  "Global Risk Appetite": (Sources.VIX,),
  "Global Credit Condition": (Sources.UST_10Y, Sources.HY_OAS),
  "FX Pass-Through": (Sources.USDMYR,),
  "Commodity": (Sources.crude_palm_oil, Sources.brent_oil),
}
ALL_PREDICTORS = [v for vlist in CHANNELS.values() for v in vlist]
DEPENDENT_VARS = [Sources.KLCI, Sources.financials, Sources.energy, Sources.industrial_products, Sources.plantation, Sources.REITs, Sources.technology]

df_monthly = pd.read_parquet(parquet_monthly).xs("change", axis=1, level=1)

def fit_hac_model(target, rhs_vars, data) -> RegressionResultsWrapper:
  formula = f"{target} ~ {' + '.join(rhs_vars)}"
  L = int(np.floor(4 * (len(data) / 100) ** (2 / 9)))
  return smf.ols(formula, data=data).fit(cov_type="HAC", cov_kwds={"maxlags": L})

def shapley_r2(y: pd.Series, X: pd.DataFrame) -> dict[str, float]:
  predictors = list(X.columns)
  p = len(predictors)
  Xc = sm.add_constant(X)
  r2_cache: dict[frozenset, float] = {frozenset(): 0.0}
  for k in range(1, p + 1):
    for combo in combinations(predictors, k):
      r2_cache[frozenset(combo)] = sm.OLS(y, Xc[["const"] + list(combo)]).fit().rsquared
  shapley = {}
  for j in predictors:
    rest = [v for v in predictors if v != j]
    total = 0.0
    for k in range(len(rest) + 1):
      weight = factorial(k) * factorial(p - k - 1) / factorial(p)
      for combo in combinations(rest, k):
        s = frozenset(combo)
        total += weight * (r2_cache[s | {j}] - r2_cache[s])
    shapley[j] = total
  return shapley

full_models, fit_dict, partialr2_dict, beta_dict = {}, {}, {}, {}
wald_dict = defaultdict(dict)

for d_var in DEPENDENT_VARS:
  data = df_monthly[[d_var] + ALL_PREDICTORS].dropna()
  m_full = fit_hac_model(d_var, ALL_PREDICTORS, data)
  full_models[d_var] = m_full

  fit_dict[d_var] = {
    ("Full R2"): m_full.rsquared,
    ("Adjusted R2"): m_full.rsquared_adj
  }

  shap = shapley_r2(data[d_var], data[ALL_PREDICTORS])
  partialr2_dict[d_var] = {}
  beta_dict[d_var] = {}
  for ch_name, ch_vars in CHANNELS.items():
    for var in ch_vars:
      partialr2_dict[d_var][(ch_name, var)] = shap[var]
      beta_dict[d_var][(ch_name, var)] = m_full.params[var]

  for ch_name, ch_vars in CHANNELS.items():
    hypothesis = ", ".join(f"{v} = 0" for v in ch_vars)
    w_res = m_full.wald_test(hypothesis, use_f=True, scalar=True)
    wald_dict[ch_name][(d_var, "F-Stat")] = float(w_res.statistic)
    wald_dict[ch_name][(d_var, "p-value")] = float(w_res.pvalue)

df_beta = pd.DataFrame(beta_dict)
df_fit = pd.DataFrame(fit_dict)
df_wald = pd.DataFrame(wald_dict).T
df_partialr2 = pd.DataFrame(partialr2_dict).div(df_fit.loc["Full R2"], axis=1) * 100

### 1. ADF (Stationary), VIF (Multicollinearity)

In [ ]:
print("Diagnostic Assumptions (ADF, VIF):")

diagnostic_dict = {}
for col in ALL_PREDICTORS + DEPENDENT_VARS:
  stat, p = adfuller(df_monthly[col].dropna(), autolag="AIC", result_object=True)[:2]
  diagnostic_dict[col] = {
    ("ADF", "score"): float(stat),
    ("ADF", "p-val"): float(p)
  }

exog_matrix = full_models[DEPENDENT_VARS[0]].model.exog
exog_names = full_models[DEPENDENT_VARS[0]].model.exog_names
for i, var_name in enumerate(exog_names):
  if var_name != "Intercept":
    vif_score = variance_inflation_factor(exog_matrix, i)
    diagnostic_dict[var_name][("VIF", "score")] = float(vif_score)

df_adf_vif = pd.DataFrame(diagnostic_dict).T
print(df_adf_vif)

### 4. MODEL FIT & VARIANCE DECOMPOSITION

In [ ]:
print("Model Fit:")
print(df_fit.to_string())
print("\nChannel Variance Decomposition (Contribution to R²):")
print(df_partialr2.groupby(level=0, sort=False).sum().to_string())

### 5. INFERENCE & FEATURE IMPORTANCE

In [ ]:
print("Channel Wald Joint Significance Tests:")
print(df_wald.to_string())

In [40]:
df_beta.index.names = df_partialr2.index.names = ['Channels', 'Variables']
df_beta.columns.name = df_fit.columns.name = df_partialr2.columns.name = 'Indices'

df_wald.index.name = 'Channels'
df_wald.columns.names = ['Indices', 'Metrics']

df_adf_vif.columns.names = ["Test", "Metrics"]
df_adf_vif.index.name = 'Variables'

analysis_dfs: dict = {
   'ADF VIF Diagnostics': {
      'df': df_adf_vif,
      'title': 'Time-Series Diagnostics (ADF Stationarity & VIF)',
      'caption': 'Pre-estimation time-series diagnostics: Augmented Dickey-Fuller (ADF) unit root tests for stationarity and Variance Inflation Factor (VIF) scores for multicollinearity.'
   },

   'Model Fit': {
      'df': df_fit,
      'title': 'Sector Model Fit & Explanatory Power (R²)',
      'caption': "Overall sector model goodness-of-fit metrics, reporting unadjusted Full R² and Adjusted R² across all sector multivariate regressions."
   },

   'Variance Decomposition': {
      'df': df_partialr2,
      'title': 'Channel Variance Decomposition (Shapley-LMG % Share)',
      'caption': "Shapley-LMG variance decomposition attributing common-size explanatory power (% share of total model R²) to each macroeconomic channel across all 2ᴾ sub-model orderings."
   },

   'Wald Joint Tests': {
      'df': df_wald,
      'title': 'Macro Channel Joint Significance (Wald F-Tests)',
      'caption': "Joint Wald F-test statistics and corresponding p-values evaluating the null hypothesis (H₀) that all predictor coefficients within a given transmission channel are simultaneously zero."
   },

   'Standardized Coefficients': {
      'df': df_beta,
      'title': 'Standardized Factor Transmission Coefficients (β)',
      'caption': "Standardized OLS regression coefficients (β) reflecting the marginal impact of macroeconomic predictors on Bursa Malaysia sector returns. Standard errors are adjusted for heteroskedasticity and autocorrelation using Newey-West (HAC) covariance estimation."
   },
}

with open(f"{PROCESSED_DATA_PATH}/analysis_dfs.pkl", "wb") as f:
   pickle.dump(analysis_dfs, f)